In [1]:
import re
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                       "datasets", "openpyxl", "tqdm"])

import os, random, time, copy
import numpy as np
from collections import Counter, defaultdict
from tqdm.auto import tqdm
import warnings; warnings.filterwarnings('ignore')
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
if device.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}, "
          f"VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

SEED = 42; random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
OUT = "/kaggle/working/"; os.makedirs(OUT, exist_ok=True)


# ─────────────────────────────────────────────────────────────────────
# 3. SLAKE  (mdwiratathya/SLAKE-vqa-english)
# ─────────────────────────────────────────────────────────────────────
# ~14,028 QA pairs (English subset) · 642 images
# Modalities: CT, MRI, X-Ray · Body parts: head/neck/chest/abdomen/pelvis
# Closed-ended (yes/no) + Open-ended (organ, modality, plane, position,
# abnormality, size, color, shape, KG-based questions)

def normalize_answer_slake(ans: str) -> str:
    """Normalize SLAKE answers."""
    ans = ans.strip().lower()
    ans = re.sub(r'[^\w\s\-/.,]', '', ans)
    ans = re.sub(r'\s+', ' ', ans).strip()

    # ── Yes / No ──
    yes_set = {'yes', 'yes.', 'yeah', 'yep', 'y', 'correct', 'true'}
    no_set  = {'no', 'no.', 'nope', 'n', 'false', 'incorrect', 'negative',
               'none', 'not sure'}
    if ans in yes_set:
        return 'yes'
    if ans in no_set:
        return 'no'

    # ── Numeric ──
    word_to_num = {'zero': '0', 'one': '1', 'two': '2', 'three': '3',
                   'four': '4', 'five': '5', 'six': '6', 'seven': '7',
                   'eight': '8', 'nine': '9', 'ten': '10'}
    if ans in word_to_num:
        return word_to_num[ans]

    # ── Modality synonyms (SLAKE has CT/MRI/X-Ray) ──
    modality_map = {
        'ct scan': 'ct', 'ct': 'ct', 'computed tomography': 'ct',
        'cat scan': 'ct',
        'mri': 'mri', 'magnetic resonance imaging': 'mri', 'mr': 'mri',
        'mri - Loss contrast': 'mri', 't1': 'mri', 't2': 'mri',
        't1-weighted': 'mri', 't2-weighted': 'mri', 'flair': 'mri',
        'x-ray': 'x-ray', 'x ray': 'x-ray', 'xray': 'x-ray',
        'radiograph': 'x-ray', 'plain film': 'x-ray',
    }
    if ans in modality_map:
        return modality_map[ans]

    # ── Plane synonyms ──
    plane_map = {
        'axial': 'axial', 'transverse': 'axial', 'horizontal': 'axial',
        'axial plane': 'axial', 'transverse plane': 'axial',
        'coronal': 'coronal', 'frontal': 'coronal', 'coronal plane': 'coronal',
        'sagittal': 'sagittal', 'sagittal plane': 'sagittal',
    }
    if ans in plane_map:
        return plane_map[ans]

    # ── Anatomical / organ synonyms (SLAKE covers head/chest/abdomen/pelvis) ──
    anatomy_map = {
        'brain': 'brain', 'cerebral': 'brain', 'cerebrum': 'brain',
        'head': 'brain', 'cranium': 'brain',
        'lung': 'lung', 'lungs': 'lung', 'pulmonary': 'lung',
        'left lung': 'left lung', 'right lung': 'right lung',
        'heart': 'heart', 'cardiac': 'heart',
        'liver': 'liver', 'hepatic': 'liver',
        'kidney': 'kidney', 'kidneys': 'kidney', 'renal': 'kidney',
        'left kidney': 'left kidney', 'right kidney': 'right kidney',
        'spleen': 'spleen', 'splenic': 'spleen',
        'pancreas': 'pancreas', 'pancreatic': 'pancreas',
        'gallbladder': 'gallbladder', 'gall bladder': 'gallbladder',
        'stomach': 'stomach', 'gastric': 'stomach',
        'bladder': 'bladder', 'urinary bladder': 'bladder',
        'spine': 'spine', 'spinal': 'spine', 'vertebral': 'spine',
        'vertebra': 'spine', 'vertebrae': 'spine',
        'chest': 'chest', 'thorax': 'chest', 'thoracic': 'chest',
        'abdomen': 'abdomen', 'abdominal': 'abdomen',
        'pelvis': 'pelvis', 'pelvic': 'pelvis',
        'neck': 'neck', 'cervical': 'neck',
    }
    if ans in anatomy_map:
        return anatomy_map[ans]

    # ── Laterality ──
    lat_map = {
        'right side': 'right', 'right-sided': 'right',
        'left side': 'left', 'left-sided': 'left',
        'both sides': 'bilateral', 'bilateral': 'bilateral', 'both': 'bilateral',
    }
    if ans in lat_map:
        return lat_map[ans]

    # ── Abnormality synonyms ──
    abnorm_map = {
        'normal': 'normal', 'no abnormality': 'normal',
        'no abnormalities': 'normal', 'no finding': 'normal',
        'no findings': 'normal', 'unremarkable': 'normal',
        'tumor': 'tumor', 'tumour': 'tumor', 'mass': 'tumor',
        'neoplasm': 'tumor',
        'inflammation': 'inflammation', 'inflamed': 'inflammation',
        'inflammatory': 'inflammation',
        'fracture': 'fracture', 'broken': 'fracture',
        'effusion': 'effusion', 'fluid': 'effusion',
        'pleural effusion': 'pleural effusion',
        'pneumonia': 'pneumonia',
        'edema': 'edema', 'oedema': 'edema', 'swelling': 'edema',
        'hemorrhage': 'hemorrhage', 'haemorrhage': 'hemorrhage',
        'bleeding': 'hemorrhage',
        'atrophy': 'atrophy', 'atrophic': 'atrophy',
        'calcification': 'calcification', 'calcified': 'calcification',
        'enlarged': 'enlargement', 'enlargement': 'enlargement',
        'hypertrophy': 'enlargement',
    }
    if ans in abnorm_map:
        return abnorm_map[ans]

    # ── Remove articles ──
    ans = re.sub(r'^(the|a|an)\s+', '', ans)
    ans = re.sub(r'\s+', ' ', ans).strip()

    return ans


# ═══════════════════════════════════════════════════════════════════════
# HYPERPARAMETERS
# ═══════════════════════════════════════════════════════════════════════
D = 256; VS = 30522; ML = 64; H = 4; DR = 0.15
CB = 3; TEL = 2; TRL = 2; FL = 4
NC = 5; RDS = 15; BS = 32; SLR = 3e-4; CLR = 3e-5; WD = 1e-4

# Adaptive CAWA Hyperparameters
CR = 0.05; CP = 0.05; CG = 0.5; CL = 0.8; CT = 1.0; CPHI = 2.0

# ═══════════════════════════════════════════════════════════════════════
# DATA
# ═══════════════════════════════════════════════════════════════════════
print("\n" + "="*60 + "\nLOADING Dataset\n" + "="*60)
from datasets import load_dataset
ds = load_dataset('mdwiratathya/SLAKE-vqa-english')

def ext(sd, n):
    sa = []
    for s in tqdm(sd, desc=n):
        try:
            img=s.get('image'); q=str(s.get('question','')); 
            a=str(s.get('answer','')).strip().lower()
            a = normalize_answer_slake(a)
            if img and q and a:
                sa.append({'image': np.array(img.convert('RGB').resize((224,224)),
                           dtype=np.float32)/255.0, 'question': q, 'answer': a})
        except: continue
    print(f"  {n}: {len(sa)}"); return sa
train_s = ext(ds['train'],'train'); test_s = ext(ds['test'],'test'); del ds
all_a = [s['answer'] for s in train_s+test_s]
av = {'<unk>':0}
for i,a in enumerate(sorted(set(all_a))): av[a] = i+1
ncls = len(av); print(f"  Classes: {ncls}")

def tok(qs):
    il,ml=[],[]
    for q in qs:
        w=q.lower().split()[:ML-2]; ids=[1]+[hash(x)%(VS-2)+2 for x in w]+[2]; m=[1.]*len(ids)
        while len(ids)<ML: ids.append(0); m.append(0.)
        il.append(ids[:ML]); ml.append(m[:ML])
    return il,ml

class VDS(Dataset):
    def __init__(s,sa,vo,aug=False):
        s.sa=sa; s.vo=vo; s.aug=aug; s.ids,s.masks=tok([x['question'] for x in sa])
    def __len__(s): return len(s.sa)
    def __getitem__(s,i):
        x=s.sa[i]; img=torch.tensor(x['image']).permute(2,0,1)
        if s.aug and random.random()>0.5: img=img.flip(-1)
        return img, torch.tensor(s.ids[i],dtype=torch.long), \
               torch.tensor(s.masks[i],dtype=torch.float32), s.vo.get(x['answer'],0)
def coll(b):
    i,d,m,l=zip(*b)
    return torch.stack(i),torch.stack(d),torch.stack(m),torch.tensor(l,dtype=torch.long)

# IID split
ia = np.random.permutation(len(train_s)); sz = len(train_s)//NC
cspl = {c: ia[c*sz:(c+1)*sz if c<NC-1 else len(train_s)].tolist() for c in range(NC)}
cld = {cid: DataLoader(VDS([train_s[i] for i in idx],av,True), batch_size=BS,
       shuffle=True, num_workers=2, pin_memory=True, collate_fn=coll)
       for cid,idx in cspl.items()}
tld = DataLoader(VDS(test_s,av), batch_size=BS, shuffle=False, num_workers=2,
                  pin_memory=True, collate_fn=coll)

# Poisoned (Client 0: 30% label flip)
pld = {}
inv_v = {v:k for k,v in av.items()}; all_l = list(av.values())
for cid, idx in cspl.items():
    cd = [train_s[i] for i in idx]
    if cid == 0:
        for j in range(len(cd)):
            if random.random()<0.3:
                orig=av.get(cd[j]['answer'],0)
                cd[j]=dict(cd[j]); cd[j]['answer']=inv_v.get(random.choice([l for l in all_l if l!=orig]),cd[j]['answer'])
    pld[cid] = DataLoader(VDS(cd,av,True), batch_size=BS, shuffle=True,
                           num_workers=2, pin_memory=True, collate_fn=coll)

# Membership inference samples
mem_idx = random.sample(range(len(train_s)), min(200,len(train_s)))
mem_s = [train_s[i] for i in mem_idx]; nmem_s = test_s[:200]

# ═══════════════════════════════════════════════════════════════════════
# SHARED BLOCKS
# ═══════════════════════════════════════════════════════════════════════
class TF(nn.Module):
    def __init__(s,d,nh=4,fr=4,dropout=.1):
        super().__init__(); s.n1=nn.LayerNorm(d); s.n2=nn.LayerNorm(d)
        s.a=nn.MultiheadAttention(d,nh,dropout=dropout,batch_first=True)
        s.f=nn.Sequential(nn.Linear(d,d*fr),nn.GELU(),nn.Dropout(dropout),nn.Linear(d*fr,d),nn.Dropout(dropout))
    def forward(s,x,mask=None):
        h=s.n1(x); k=(mask==0) if mask is not None else None
        h,_=s.a(h,h,h,key_padding_mask=k); x=x+h; return x+s.f(s.n2(x))

class VE(nn.Module):
    def __init__(s,d=256):
        super().__init__()
        s.c1=nn.Conv2d(3,32,7,2,3,bias=False); s.b1=nn.BatchNorm2d(32); s.p1=nn.MaxPool2d(3,2,1)
        s.c2=nn.Conv2d(32,64,3,2,1,bias=False); s.b2=nn.BatchNorm2d(64)
        s.c3=nn.Conv2d(64,128,3,2,1,bias=False); s.b3=nn.BatchNorm2d(128)
        s.c4=nn.Conv2d(128,d,3,2,1,bias=False); s.b4=nn.BatchNorm2d(d); s.n=nn.LayerNorm(d)
    def forward(s,x):
        h=s.p1(F.silu(s.b1(s.c1(x)))); h=F.silu(s.b2(s.c2(h)))
        h=F.silu(s.b3(s.c3(h))); h=F.silu(s.b4(s.c4(h)))
        B,C,H,W=h.shape; return s.n(h.permute(0,2,3,1).reshape(B,H*W,C))

class TE(nn.Module):
    def __init__(s,vs=30522,d=256,nl=2,nh=4,ml=64,do=.1):
        super().__init__(); s.te=nn.Embedding(vs,d); s.pe=nn.Parameter(torch.randn(1,ml,d)*.02)
        s.en=nn.LayerNorm(d); s.ed=nn.Dropout(do)
        s.bl=nn.ModuleList([TF(d,nh,dropout=do) for _ in range(nl)]); s.fn=nn.LayerNorm(d)
    def forward(s,ids,mask=None):
        L=ids.shape[1]; x=s.te(ids)+s.pe[:,:L,:]; x=s.ed(s.en(x))
        for b in s.bl: x=b(x,mask=mask)
        return s.fn(x)

class CA(nn.Module):
    def __init__(s,c,r=8): super().__init__(); s.f1=nn.Linear(c,c//r,bias=False); s.f2=nn.Linear(c//r,c,bias=False)
    def forward(s,x): a=x.mean([1,2],keepdim=True); m=x.amax([1,2],keepdim=True); return x*torch.sigmoid(s.f2(F.silu(s.f1(a)))+s.f2(F.silu(s.f1(m))))

class SA(nn.Module):
    def __init__(s): super().__init__(); s.c1=nn.Conv2d(2,8,3,padding=1,bias=False); s.c2=nn.Conv2d(2,8,3,padding=2,dilation=2,bias=False); s.fu=nn.Conv2d(16,1,1,bias=False)
    def forward(s,x): xp=x.permute(0,3,1,2); a=xp.mean(1,keepdim=True); m=xp.amax(1,keepdim=True); c=torch.cat([a,m],1); return x*torch.sigmoid(s.fu(torch.cat([s.c1(c),s.c2(c)],1))).permute(0,2,3,1)

class CB(nn.Module):
    def __init__(s,c): super().__init__(); s.ca=CA(c); s.sa=SA(); s.f=nn.Sequential(nn.Linear(c,c*2),nn.GELU(),nn.Linear(c*2,c)); s.n1=nn.LayerNorm(c); s.n2=nn.LayerNorm(c)
    def forward(s,t): B,N,C=t.shape; sp=s.sa(s.ca(t.reshape(B,7,7,C))); t=s.n1(t+sp.reshape(B,N,C)); return s.n2(t+s.f(t))

class FuL(nn.Module):
    def __init__(s,d,nh,do):
        super().__init__()
        s.v2t=nn.MultiheadAttention(d,nh,dropout=do,batch_first=True); s.vn=nn.LayerNorm(d); s.vf=nn.Sequential(nn.Linear(d,d*4),nn.GELU(),nn.Dropout(do),nn.Linear(d*4,d)); s.vfn=nn.LayerNorm(d)
        s.t2v=nn.MultiheadAttention(d,nh,dropout=do,batch_first=True); s.tn=nn.LayerNorm(d); s.tf=nn.Sequential(nn.Linear(d,d*4),nn.GELU(),nn.Dropout(do),nn.Linear(d*4,d)); s.tfn=nn.LayerNorm(d)
    def forward(s,v,t,k=None):
        o,_=s.v2t(v,t,t,key_padding_mask=k); v=s.vn(v+o); v=s.vfn(v+s.vf(v))
        o,_=s.t2v(t,v,v); t=s.tn(t+o); t=s.tfn(t+s.tf(t)); return v,t

class ClassificationHead(nn.Module):
    """Shared head logic for rigorous comparison."""
    def __init__(self, nc):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(D, D), nn.GELU(), nn.Dropout(DR), nn.Linear(D, nc))
    def forward(self, x): return self.net(x)

# ═══════════════════════════════════════════════════════════════════════
# USplit (U-Shape)
# ═══════════════════════════════════════════════════════════════════════
class UC(nn.Module):
    """USplit Client: encoders (bottom) + classifier (tail)."""
    def __init__(self, nc):
        super().__init__()
        self.ve=VE(D); self.te=TE(VS,D,TEL,H,ML,DR)
        self.head=ClassificationHead(nc)
    def encode(self, img, ids, mask): return self.ve(img), self.te(ids, mask=mask)
    def classify(self, fused): return self.head(fused)
    @property
    def smashed_dim(self): return D

class US(nn.Module):
    """USplit Server: middle only."""
    def __init__(self):
        super().__init__()
        self.vr=nn.ModuleList([CB(D) for _ in range(3)])
        self.tr=nn.ModuleList([TF(D,H,dropout=DR) for _ in range(TRL)]); self.trn=nn.LayerNorm(D)
        self.qa=nn.MultiheadAttention(D,H,dropout=DR,batch_first=True); self.qg=nn.Linear(D,D); self.qn=nn.LayerNorm(D)
        self.fl=nn.ModuleList([FuL(D,H,DR) for _ in range(FL)])
        self.pq=nn.Parameter(torch.randn(1,1,D)*.02); self.pa=nn.MultiheadAttention(D,H,dropout=DR,batch_first=True); self.pn=nn.LayerNorm(D)
    def forward(self, v, t, mask):
        for b in self.vr: v=b(v)
        for b in self.tr: t=b(t,mask=mask)
        t=self.trn(t); qc=t[:,0:1,:].expand(-1,v.shape[1],-1)
        ao,_=self.qa(qc,v,v); g=torch.sigmoid(self.qg(ao)); v=self.qn(v+v*g+ao*(1-g))
        k=(mask==0)
        for f in self.fl: v,t=f(v,t,k=k)
        c=torch.cat([v,t],1); B=c.shape[0]; pq=self.pq.expand(B,-1,-1)
        p,_=self.pa(pq,c,c); return self.pn(pq+p).squeeze(1)

# ═══════════════════════════════════════════════════════════════════════
# BiCSL (Standard Split)
# ═══════════════════════════════════════════════════════════════════════
class BC(nn.Module):
    """BiCSL Client: encoders only (Outputs native D dim)."""
    def __init__(self):
        super().__init__()
        self.ve=VE(D); self.te=TE(VS,D,TEL,H,ML,DR)
    def encode(self, img, ids, mask): return self.ve(img), self.te(ids, mask=mask)
    @property
    def smashed_dim(self): return D

class BS(nn.Module):
    """BiCSL Server: Identical processing capacity to USplit."""
    def __init__(self, nc):
        super().__init__()
        self.vr=nn.ModuleList([CB(D) for _ in range(3)])
        self.tr=nn.ModuleList([TF(D,H,dropout=DR) for _ in range(TRL)]); self.trn=nn.LayerNorm(D)
        self.qa=nn.MultiheadAttention(D,H,dropout=DR,batch_first=True); self.qg=nn.Linear(D,D); self.qn=nn.LayerNorm(D)
        self.fl=nn.ModuleList([FuL(D,H,DR) for _ in range(FL)])
        self.pq=nn.Parameter(torch.randn(1,1,D)*.02); self.pa=nn.MultiheadAttention(D,H,dropout=DR,batch_first=True); self.pn=nn.LayerNorm(D)
        self.head = ClassificationHead(nc)
    def forward(self, v, t, mask):
        for b in self.vr: v=b(v)
        for b in self.tr: t=b(t,mask=mask)
        t=self.trn(t); qc=t[:,0:1,:].expand(-1,v.shape[1],-1)
        ao,_=self.qa(qc,v,v); g=torch.sigmoid(self.qg(ao)); v=self.qn(v+v*g+ao*(1-g))
        k=(mask==0)
        for f in self.fl: v,t=f(v,t,k=k)
        c=torch.cat([v,t],1); B=c.shape[0]; pq=self.pq.expand(B,-1,-1)
        p,_=self.pa(pq,c,c); f_fused=self.pn(pq+p).squeeze(1)
        return self.head(f_fused)

# ═══════════════════════════════════════════════════════════════════════
# TRAINING HELPER (Fair joint training)
# ═══════════════════════════════════════════════════════════════════════
def train(client, server, loaders, tl, name, cawa=False, is_u=False, rds=RDS):
    crit=nn.CrossEntropyLoss()
    sopt=torch.optim.AdamW(server.parameters(),lr=SLR,weight_decay=WD)
    copt=torch.optim.AdamW(client.parameters(),lr=CLR,weight_decay=WD)
    
    # Adaptive CAWA Tracking
    reps={c:0.0 for c in range(NC)}
    streaks={c:0 for c in range(NC)}
    scores={c:1.0 for c in range(NC)}
    hist={'round':[],'test_acc':[],'train_loss':[],'cawa':defaultdict(list)}

    for rnd in range(1,rds+1):
        server.train(); client.train()
        rl=[]; rc,rt=0,0; rg=[]; rci=[]

        if cawa:
            max_rep = max(reps.values()) if reps else 0.0
            scores = {c: float(np.exp((reps[c] - max_rep) / CT)) for c in reps}

        for cid in range(NC):
            cl,cc,ct=0.,0,0; cg=[]
            for im,ids,msk,lb in loaders[cid]:
                im,ids,msk,lb=im.to(device),ids.to(device),msk.to(device),lb.to(device)
                sopt.zero_grad(); copt.zero_grad()
                
                v,t=client.encode(im,ids,msk)
                if is_u: logits=client.classify(server(v,t,msk))
                else: logits=server(v,t,msk)
                
                loss=crit(logits,lb)
                if cawa: loss=loss*scores[cid]
                loss.backward()
                
                if cawa:
                    sg=[p.grad.detach().cpu().numpy().copy() for p in server.parameters() if p.grad is not None]
                    cg.append(sg)
                nn.utils.clip_grad_norm_(server.parameters(),1.0); sopt.step()
                nn.utils.clip_grad_norm_(client.parameters(),1.0); copt.step()
                
                cl+=crit(logits.detach(),lb).item()*lb.size(0)
                cc+=(logits.argmax(-1)==lb).sum().item(); ct+=lb.size(0)
            rl.append(cl/max(ct,1)); rc+=cc; rt+=ct
            if cawa and cg:
                nb=len(cg); rg.append([sum(cg[b][p] for b in range(nb))/nb for p in range(len(cg[0]))]); rci.append(cid)

        # Adaptive CAWA update
        if cawa and len(rg)>=2:
            flat_grads = [np.concatenate([g.ravel() for g in gl]) for gl in rg]
            stack_g = np.stack(flat_grads)
            
            # Precompute pairwise cosine similarity matrix
            norms = np.linalg.norm(stack_g, axis=1, keepdims=True)
            stack_g_norm = stack_g / (norms + 1e-8)
            cos_sim = np.dot(stack_g_norm, stack_g_norm.T)
            
            current_weights = [scores[ci] for ci in rci]
            sims = []
            N = len(rg)
            for k in range(N):
                w_sum = sum(current_weights[j] for j in range(N) if j != k)
                w_sum = max(w_sum, 1e-8)
                # Reputation-weighted similarity
                sim_k = sum((current_weights[j] / w_sum) * cos_sim[k, j] for j in range(N) if j != k)
                sims.append(sim_k)
                
            # Adaptive Thresholds
            mu = np.mean(sims); sigma = np.std(sims)
            tau_plus = mu + CL * sigma
            tau_minus = mu - CL * sigma
            rho = (rnd / rds) ** CPHI # Temporal Scaling
            
            for ii, ci in enumerate(rci):
                sim = sims[ii]
                if sim > tau_plus:
                    streaks[ci] = max(1, streaks[ci] + 1)
                    reps[ci] += rho * CR * np.exp(CG * (streaks[ci] - 1))
                elif sim < tau_minus:
                    streaks[ci] = min(-1, streaks[ci] - 1)
                    reps[ci] -= rho * CP * np.exp(CG * (abs(streaks[ci]) - 1))
                else:
                    streaks[ci] = 0
                    
        for cid in range(NC): 
            if cawa: hist['cawa'][cid].append(round(reps[cid],3))
            else: hist['cawa'][cid].append(1.0)

        # Eval
        server.eval(); client.eval()
        with torch.no_grad():
            c2,t2=0,0
            for im,ids,msk,lb in tl:
                im,ids,msk,lb=im.to(device),ids.to(device),msk.to(device),lb.to(device)
                v,t=client.encode(im,ids,msk)
                lo=client.classify(server(v,t,msk)) if is_u else server(v,t,msk)
                c2+=(lo.argmax(-1)==lb).sum().item(); t2+=lb.size(0)
        tea=100*c2/max(t2,1); al=np.mean(rl)
        hist['round'].append(rnd); hist['test_acc'].append(round(tea,2)); hist['train_loss'].append(round(al,4))
        
        sc=" | Reps: " + " ".join(f"C{c}:{reps[c]:.2f}" for c in range(NC)) if cawa else ""
        print(f"  [{name}] R{rnd:02d}  Loss:{al:.4f}  Acc:{tea:.1f}%{sc}")
    return hist

# ═══════════════════════════════════════════════════════════════════════
# ATTACKS
# ═══════════════════════════════════════════════════════════════════════
def atk_label_inf(client, server, ld, is_u, name):
    print(f"\n  [{name}] Label Inference Attack (on smashed features)")
    client.eval(); server.eval()
    feats,labs=[],[]
    with torch.no_grad():
        for im,ids,msk,lb in ld:
            im,ids,msk=im.to(device),ids.to(device),msk.to(device)
            v,t=client.encode(im,ids,msk)
            feats.append(torch.cat([v.mean(1),t.mean(1)],dim=-1).cpu()); labs.append(lb)
    feats=torch.cat(feats); labs=torch.cat(labs)
    sp=int(0.7*len(feats)); f_dim=feats.shape[-1]

    atk=nn.Sequential(nn.Linear(f_dim,D),nn.ReLU(),nn.Dropout(.3),nn.Linear(D,ncls)).to(device)
    opt=torch.optim.Adam(atk.parameters(),lr=1e-3)

    for _ in range(50):
        atk.train(); idx=torch.randperm(sp)
        for i in range(0,sp,64):
            b=idx[i:i+64]; f=feats[b].to(device); l=labs[b].to(device)
            opt.zero_grad(); F.cross_entropy(atk(f),l).backward(); opt.step()

    atk.eval()
    with torch.no_grad(): acc=100*(atk(feats[sp:].to(device)).argmax(-1).cpu()==labs[sp:]).float().mean().item()
    print(f"    Attacker acc: {acc:.1f}% (random: {100/ncls:.1f}%)"); del atk; return acc

def atk_model_inv(client, ld, name, sdim, n_ep=30):
    print(f"\n  [{name}] Model Inversion (smashed_dim={sdim})")
    client.eval()
    dec=nn.Sequential(
        nn.Linear(sdim,512), nn.ReLU(),
        nn.Unflatten(1,(512,1,1)),
        nn.ConvTranspose2d(512,256,7,1,0), nn.BatchNorm2d(256), nn.ReLU(),
        nn.ConvTranspose2d(256,128,4,2,1), nn.BatchNorm2d(128), nn.ReLU(),
        nn.ConvTranspose2d(128,64,4,2,1), nn.BatchNorm2d(64), nn.ReLU(),
        nn.ConvTranspose2d(64,32,4,2,1), nn.BatchNorm2d(32), nn.ReLU(),
        nn.ConvTranspose2d(32,16,4,2,1), nn.BatchNorm2d(16), nn.ReLU(),
        nn.ConvTranspose2d(16,3,4,2,1), nn.Sigmoid()
    ).to(device)
    opt=torch.optim.Adam(dec.parameters(),lr=1e-3)
    final_mse=0.
    for ep in range(1,n_ep+1):
        dec.train(); tm=0.; nb=0
        for im,ids,msk,_ in ld:
            im,ids,msk=im.to(device),ids.to(device),msk.to(device)
            with torch.no_grad():
                v,t=client.encode(im,ids,msk)
            smashed=v.mean(1)
            recon=dec(smashed)
            loss=F.mse_loss(recon,im)
            opt.zero_grad(); loss.backward(); opt.step()
            tm+=loss.item(); nb+=1
        final_mse=tm/nb
    psnr=10*np.log10(1./max(final_mse,1e-10))
    print(f"    MSE: {final_mse:.6f}, PSNR: {psnr:.2f} dB"); del dec; return final_mse,psnr

def atk_membership(client, server, is_u, name):
    print(f"\n  [{name}] Membership Inference")
    client.eval(); server.eval()
    def get_conf(samples):
        ds=VDS(samples,av); ld=DataLoader(ds,batch_size=32,shuffle=False,collate_fn=coll)
        confs=[]
        with torch.no_grad():
            for im,ids,msk,lb in ld:
                im,ids,msk,lb=im.to(device),ids.to(device),msk.to(device),lb.to(device)
                v,t=client.encode(im,ids,msk)
                lo=client.classify(server(v,t,msk)) if is_u else server(v,t,msk)
                pr=F.softmax(lo,dim=-1)
                for j in range(lb.size(0)): confs.append(pr[j,lb[j]].item())
        return confs
    mc=get_conf(mem_s); nc_=get_conf(nmem_s)
    ac=mc+nc_; al_=[1]*len(mc)+[0]*len(nc_)
    th=np.median(ac); preds=[1 if c>th else 0 for c in ac]
    acc=100*sum(p==l for p,l in zip(preds,al_))/len(al_)
    print(f"    Attack acc: {acc:.1f}% (50%=random)"); return acc,np.mean(mc),np.mean(nc_)

def atk_grad_leak(client, server, ld, is_u, name):
    print(f"\n  [{name}] Gradient Leakage")
    crit=nn.CrossEntropyLoss(); client.train(); server.train()
    sn,cn=[],[]
    for i,(im,ids,msk,lb) in enumerate(ld):
        if i>=5: break
        im,ids,msk,lb=im.to(device),ids.to(device),msk.to(device),lb.to(device)
        server.zero_grad(); client.zero_grad()
        v,t=client.encode(im,ids,msk)
        lo=client.classify(server(v,t,msk)) if is_u else server(v,t,msk)
        crit(lo,lb).backward()
        sn.append(sum(p.grad.norm().item()**2 for p in server.parameters() if p.grad is not None)**.5)
        cn.append(sum(p.grad.norm().item()**2 for p in client.parameters() if p.grad is not None)**.5)
    as_=np.mean(sn); ac_=np.mean(cn); ratio=as_/(as_+ac_+1e-8)
    sp=sum(p.numel() for p in server.parameters())
    tp=sp+sum(p.numel() for p in client.parameters())
    exp=100*sp/tp
    print(f"    Server grad fraction: {ratio:.4f}, exposure: {exp:.1f}%"); return ratio,exp

# ═══════════════════════════════════════════════════════════════════════
# RUN ALL
# ═══════════════════════════════════════════════════════════════════════
print("\n" + "="*70 + "\n   EVAL A: Clean Accuracy\n" + "="*70)
uc=UC(ncls).to(device); us=US().to(device)
hoc=train(uc,us,cld,tld,"USplit",cawa=True,is_u=True)
bc=BC().to(device); bs_m=BS(ncls).to(device)
hbc=train(bc,bs_m,cld,tld,"BiCSL",is_u=False)

print("\n" + "="*70 + "\n   EVAL B: Byzantine Robustness\n" + "="*70)
uc2=UC(ncls).to(device); us2=US().to(device)
hob=train(uc2,us2,pld,tld,"USplit+Poison",cawa=True,is_u=True)
bc2=BC().to(device); bs2=BS(ncls).to(device)
hbb=train(bc2,bs2,pld,tld,"BiCSL+Poison",is_u=False)

print("\n" + "="*70 + "\n   EVAL C: Label Inference\n" + "="*70)
li_o=atk_label_inf(uc,us,tld,True,"USplit")
li_b=atk_label_inf(bc,bs_m,tld,False,"BiCSL")

print("\n" + "="*70 + "\n   EVAL D: Model Inversion\n" + "="*70)
mi_o_mse,mi_o_psnr=atk_model_inv(uc,tld,"USplit",sdim=D)
mi_b_mse,mi_b_psnr=atk_model_inv(bc,tld,"BiCSL",sdim=D)

print("\n" + "="*70 + "\n   EVAL E: Membership Inference\n" + "="*70)
me_o_acc,me_o_m,me_o_nm=atk_membership(uc,us,True,"USplit")
me_b_acc,me_b_m,me_b_nm=atk_membership(bc,bs_m,False,"BiCSL")

print("\n" + "="*70 + "\n   EVAL F: Gradient Leakage\n" + "="*70)
gl_o_r,gl_o_e=atk_grad_leak(uc,us,tld,True,"USplit")
gl_b_r,gl_b_e=atk_grad_leak(bc,bs_m,tld,False,"BiCSL")

# ═══════════════════════════════════════════════════════════════════════
# RESULTS
# ═══════════════════════════════════════════════════════════════════════
oc=hoc['test_acc'][-1]; bc_v=hbc['test_acc'][-1]
ob=hob['test_acc'][-1]; bb=hbb['test_acc'][-1]
nuc=sum(p.numel() for p in uc.parameters()); nus=sum(p.numel() for p in us.parameters())
nbc=sum(p.numel() for p in bc.parameters()); nbs=sum(p.numel() for p in bs_m.parameters())
co=(49*D+ML*D)*2+D*2; cb=(49*D+ML*D)+1

print("\n" + "="*70 + "\n   COMPREHENSIVE COMPARISON\n" + "="*70)
comp = [
    ("Clean Test Accuracy (%)",      oc, bc_v, False),
    ("Byzantine Test Accuracy (%)",  ob, bb, False),
    ("Accuracy Drop (%)",            round(oc-ob,2), round(bc_v-bb,2), True),
    ("Label Inference Acc (%)",      round(li_o,1), round(li_b,1), True),
    ("Model Inversion PSNR (dB)",   round(mi_o_psnr,2), round(mi_b_psnr,2), True),
    ("Membership Inference (%)",     round(me_o_acc,1), round(me_b_acc,1), True),
    ("Server Grad Fraction",        round(gl_o_r,4), round(gl_b_r,4), True),
    ("Server Grad Exposure (%)",    round(gl_o_e,1), round(gl_b_e,1), True),
    ("Client Params",               nuc, nbc, None),
    ("Server Params",               nus, nbs, None),
    ("Comm/sample (floats)",        co, cb, True),
]
strs = [
    ("Label Privacy",     "YES", "NO (server sees labels)", "Ours"),
    ("Topology",          "U-Shaped", "Standard Split", "-"),
    ("CAWA Defense",      "YES", "NO", "Ours"),
    ("Loss Computation",  "Client-side", "Server-side", "Ours"),
]

print(f"\n{'Metric':<35} {'USplit+CAWA':>18} {'BiCSL':>18} {'Winner':>8}")
print("-"*79)
for m,ov,bv,lo in comp:
    if lo is None: w="-"
    elif lo: w="Ours" if ov<bv else ("BiCSL" if bv<ov else "Tie")
    else: w="Ours" if ov>bv else ("BiCSL" if bv>ov else "Tie")
    print(f"{m:<35} {str(ov):>18} {str(bv):>18} {w:>8}")
print()
for m,ov,bv,w in strs: print(f"{m:<35} {ov:>18} {bv:>18} {w:>8}")

# ═══════════════════════════════════════════════════════════════════════
# EXCEL
# ═══════════════════════════════════════════════════════════════════════
import openpyxl; from openpyxl.styles import Font,PatternFill,Alignment
wb=openpyxl.Workbook()
hf=Font(name='Arial',bold=True,size=11,color='FFFFFF')
hfi=PatternFill(start_color='1B4F72',end_color='1B4F72',fill_type='solid')
grn=PatternFill(start_color='D5F5E3',end_color='D5F5E3',fill_type='solid')
red=PatternFill(start_color='FADBD8',end_color='FADBD8',fill_type='solid')

# Sheet 1: Per round
ws=wb.active; ws.title="Per Round"
hdrs=['Round','Ours Clean','Ours Loss','BiCSL Clean','BiCSL Loss',
      'Ours Byz','Ours Byz Loss','BiCSL Byz','BiCSL Byz Loss']
for c in range(NC): hdrs.append(f'C{c} CAWA')
for c,h in enumerate(hdrs,1):
    cl=ws.cell(row=1,column=c,value=h); cl.font=hf; cl.fill=hfi; cl.alignment=Alignment(horizontal='center')
for i in range(RDS):
    r=i+2; ws.cell(row=r,column=1,value=i+1)
    ws.cell(row=r,column=2,value=hoc['test_acc'][i]); ws.cell(row=r,column=3,value=hoc['train_loss'][i])
    ws.cell(row=r,column=4,value=hbc['test_acc'][i]); ws.cell(row=r,column=5,value=hbc['train_loss'][i])
    ws.cell(row=r,column=6,value=hob['test_acc'][i]); ws.cell(row=r,column=7,value=hob['train_loss'][i])
    ws.cell(row=r,column=8,value=hbb['test_acc'][i]); ws.cell(row=r,column=9,value=hbb['train_loss'][i])
    for c in range(NC): ws.cell(row=r,column=10+c,value=hob['cawa'][c][i])

# Sheet 2: Comparison
ws2=wb.create_sheet("Comparison")
hdrs2=['Metric','USplit+CAWA','BiCSL','Winner','Category']
for c,h in enumerate(hdrs2,1):
    cl=ws2.cell(row=1,column=c,value=h); cl.font=hf; cl.fill=hfi; cl.alignment=Alignment(horizontal='center')
all_m=[]
for m,ov,bv,lo in comp:
    if lo is None: w="-"
    elif lo: w="Ours" if ov<bv else ("BiCSL" if bv<ov else "Tie")
    else: w="Ours" if ov>bv else ("BiCSL" if bv>ov else "Tie")
    cat="Performance" if "Accuracy" in m else ("Security" if any(k in m for k in ["Inference","Inversion","Grad","Label"]) else "Efficiency")
    all_m.append((m,ov,bv,w,cat))
for m,ov,bv,w in strs: all_m.append((m,ov,bv,w,"Architecture"))
for i,(m,ov,bv,w,cat) in enumerate(all_m):
    r=i+2; ws2.cell(row=r,column=1,value=m); ws2.cell(row=r,column=2,value=ov)
    ws2.cell(row=r,column=3,value=bv); ws2.cell(row=r,column=4,value=w); ws2.cell(row=r,column=5,value=cat)
    if w=="Ours": ws2.cell(row=r,column=4).fill=grn
    elif w=="BiCSL": ws2.cell(row=r,column=4).fill=red

# Sheet 3: Attack Details
ws3=wb.create_sheet("Attack Details")
atk_d=[
    ("Attack","USplit Result","BiCSL Result","Interpretation"),
    ("A. Clean Acc",f"{oc:.2f}%",f"{bc_v:.2f}%","Server capacities now equalized"),
    ("B. Byz Acc",f"{ob:.2f}%",f"{bb:.2f}%","CAWA downweights malicious client"),
    ("B. Acc Drop",f"{oc-ob:.2f}%",f"{bc_v-bb:.2f}%","Lower=more robust under attack"),
    ("C. Label Inf",f"{li_o:.1f}%",f"{li_b:.1f}%", "Evaluated strictly on smashed feature leakage"),
    ("D. Inv MSE",f"{mi_o_mse:.6f}",f"{mi_b_mse:.6f}","Higher MSE=harder to reconstruct"),
    ("D. Inv PSNR",f"{mi_o_psnr:.2f}dB",f"{mi_b_psnr:.2f}dB","Lower=better privacy"),
    ("E. Mem Inf",f"{me_o_acc:.1f}%",f"{me_b_acc:.1f}%","Closer to 50%=better privacy"),
    ("F. Grad Frac",f"{gl_o_r:.4f}",f"{gl_b_r:.4f}","Lower=less server leakage"),
    ("F. Grad Exp",f"{gl_o_e:.1f}%",f"{gl_b_e:.1f}%","Lower=less exposure"),
]
for c,h in enumerate(atk_d[0],1):
    cl=ws3.cell(row=1,column=c,value=h); cl.font=hf; cl.fill=hfi; cl.alignment=Alignment(horizontal='center')
for i,rd in enumerate(atk_d[1:],2):
    for c,v in enumerate(rd,1):
        ws3.cell(row=i,column=c,value=v)
        if c==1: ws3.cell(row=i,column=c).font=Font(bold=True,name='Arial')

for s in [ws,ws2,ws3]:
    for col in s.columns: s.column_dimensions[col[0].column_letter].width=max(len(str(c.value or '')) for c in col)+2

p=f"{OUT}/usplit_vs_bicsl_results.xlsx"; wb.save(p)
print(f"\nSaved → {p}\nDONE!")

Device: cuda
GPU: Tesla T4, VRAM: 15.6 GB

LOADING Dataset


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00002.parquet:   0%|          | 0.00/31.1M [00:00<?, ?B/s]

data/train-00001-of-00002.parquet:   0%|          | 0.00/12.2M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/8.34M [00:00<?, ?B/s]

...

Generating test split:   0%|          | 0/1061 [00:00<?, ? examples/s]

train:   0%|          | 0/4919 [00:00<?, ?it/s]

  train: 4919


test:   0%|          | 0/1061 [00:00<?, ?it/s]

  test: 1061
  Classes: 218

   EVAL A: Clean Accuracy
  [USplit] R01  Loss:3.4441  Acc:48.8% | Reps: C0:-0.00 C1:0.00 C2:0.00 C3:0.00 C4:0.00
  [USplit] R02  Loss:2.1096  Acc:56.1% | Reps: C0:0.00 C1:0.00 C2:0.00 C3:-0.00 C4:0.00
  [USplit] R03  Loss:1.6611  Acc:61.5% | Reps: C0:0.00 C1:0.00 C2:-0.00 C3:-0.00 C4:0.00
  [USplit] R04  Loss:1.4059  Acc:64.3% | Reps: C0:0.01 C1:0.00 C2:-0.00 C3:-0.01 C4:0.00
  [USplit] R05  Loss:1.2596  Acc:67.5% | Reps: C0:0.04 C1:0.00 C2:-0.01 C3:-0.01 C4:0.00
  [USplit] R06  Loss:1.1303  Acc:69.2% | Reps: C0:0.10 C1:0.00 C2:-0.01 C3:-0.02 C4:0.00
  [USplit] R07  Loss:1.0397  Acc:70.3% | Reps: C0:0.23 C1:-0.01 C2:-0.01 C3:-0.02 C4:0.00
  [USplit] R08  Loss:0.9496  Acc:69.7% | Reps: C0:0.23 C1:-0.03 C2:-0.01 C3:-0.02 C4:0.02
  [USplit] R09  Loss:0.8866  Acc:70.2% | Reps: C0:0.25 C1:-0.03 C2:-0.01 C3:-0.04 C4:0.05
  [USplit] R10  Loss:0.8366  Acc:72.9% | Reps: C0:0.25 C1:-0.06 C2:-0.03 C3:-0.04 C4:0.11
  [USplit] R11  Loss:0.7860  Acc:72.6% | Reps: C0:0.2